# Train Raikiri on a Colab T4

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antialberteinstein/kessie-mlops-lab/blob/main/notebooks/train-raikiri.ipynb)

Select **Runtime → Change runtime type → T4 GPU**, then choose **Run all**. After training, the notebook generates Vietnamese text samples, shows training metrics and samples in TensorBoard, and uploads the model to a public Hugging Face repository named `<your-username>/raikiri`. Add a write-enabled `HF_TOKEN` in Colab's **Secrets** panel and grant this notebook access. If the secret is unavailable, the notebook securely prompts for the token.

In [ ]:
!git clone https://github.com/antialberteinstein/kessie-mlops-lab.git
%cd /content/kessie-mlops-lab
!python -m pip install -q -r projects/raikiri/requirements.txt

In [ ]:
import torch

assert torch.cuda.is_available(), "Select a T4 GPU runtime before training."
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

log_dir = Path('artifacts/raikiri/tensorboard')
log_dir.mkdir(parents=True, exist_ok=True)
%load_ext tensorboard
%tensorboard --logdir artifacts/raikiri/tensorboard

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, 'projects/raikiri/train.py'], check=True)

## Generate and log sample text

These samples are generated from the final saved model. They also appear in TensorBoard's **Text** tab and are uploaded with the model.

In [ ]:
import os

os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('USE_FLAX', '0')
sys.path.insert(0, str(Path('projects/raikiri').resolve()))
from config import DATA, MODEL, TRAIN
from model import load_trained_model
from torch.utils.tensorboard import SummaryWriter
from transformers import AutoTokenizer

model_dir = Path(TRAIN.output_dir)
tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = load_trained_model(tokenizer, model_dir, MODEL, DATA).to(device='cuda', dtype=torch.float16)
model.eval()

prompts = [
    'Việt Nam là một quốc gia',
    'Khoa học và công nghệ đang',
    'Lịch sử của Hà Nội',
]
samples = []
with torch.inference_mode():
    for prompt in prompts:
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        output = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            use_cache=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        generated = tokenizer.decode(output[0], skip_special_tokens=True)
        samples.append((prompt, generated))
        print(f'Prompt: {prompt}\nGenerated: {generated}\n')

sample_text = '# Raikiri sample generations\n\n' + '\n\n'.join(
    f'## Example {index}\n\n**Prompt:** {prompt}\n\n**Generated:** {generated}'
    for index, (prompt, generated) in enumerate(samples, start=1)
) + '\n'
(model_dir / 'sample_generations.md').write_text(sample_text, encoding='utf-8')
with SummaryWriter(log_dir=str(log_dir)) as writer:
    for index, (prompt, generated) in enumerate(samples, start=1):
        writer.add_text(f'samples/example_{index}', f'Prompt: {prompt}\n\nGenerated: {generated}', 0)

model_card = (
    '---\nlanguage: vi\nlibrary_name: transformers\ntags:\n- text-generation\n---\n\n'
    '# Raikiri\n\n'
    'A compact Vietnamese causal language model trained with the Raikiri project.\n\n'
    'The checkpoint uses the pinned Intern-S2-Mobius architecture and a forward adapter. '
    'Load it with the project loader at '
    'https://github.com/antialberteinstein/kessie-mlops-lab/tree/main/projects/raikiri '
    'after downloading this repository. See `sample_generations.md` for examples.\n'
)
(model_dir / 'README.md').write_text(model_card, encoding='utf-8')

del model
torch.cuda.empty_cache()

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

tags = EventAccumulator(str(log_dir)).Reload().Tags()
print('TensorBoard scalars:', tags['scalars'])
print('TensorBoard texts:', tags['tensors'])
assert tags['scalars'], 'No scalar logs were written by training.'
assert any(tag.startswith('samples/') for tag in tags['tensors']), 'No sample texts were logged.'
%tensorboard --logdir artifacts/raikiri/tensorboard

## Upload to Hugging Face

The token is read from the Colab secret `HF_TOKEN`; if that secret is missing or inaccessible, enter a write-enabled token in the hidden prompt. Only files in the final model directory are uploaded, excluding intermediate checkpoints.

In [ ]:
from getpass import getpass
from google.colab import userdata
from huggingface_hub import HfApi

try:
    hf_token = (userdata.get('HF_TOKEN') or '').strip()
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    hf_token = None
if not hf_token:
    hf_token = getpass('Enter your Hugging Face write token (HF_TOKEN): ').strip()
if not hf_token:
    raise ValueError('A Hugging Face write token is required to upload the model.')

api = HfApi(token=hf_token)
username = api.whoami()['name']
repo_id = f'{username}/raikiri'
model_files = [path.name for path in model_dir.iterdir() if path.is_file()]
assert 'model.safetensors' in model_files, 'Final model weights are missing.'
api.create_repo(repo_id=repo_id, private=False, exist_ok=True)
api.upload_folder(
    repo_id=repo_id,
    folder_path=model_dir,
    repo_type='model',
    allow_patterns=model_files,
    commit_message='Upload trained Raikiri model and sample generations',
)
print(f'Uploaded model: https://huggingface.co/{repo_id}')